In [1]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"    # If not used this line, retrieval.invoke() will crash the kernal

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_google_genai import ChatGoogleGenerativeAI

from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

import asyncio
from tools import google_search, weather_search, fetch_train_details

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [4]:
load_dotenv(".env.chatbot")

True

In [77]:
os.environ["LANGSMITH_PROJECT"] = "LangGraph_RAG"

In [78]:
model = ChatOpenAI(model = 'gpt-3.5-turbo')

In [79]:
# Creating loader object
loader = PyPDFLoader("MCP illustrated guidebook.pdf")

# Loading the pdf
docs = loader.load()

docs

Ignoring wrong pointing object 567 0 (offset 0)


[Document(metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250626122824Z00'00'", 'title': 'MCP book', 'moddate': "D:20250626122824Z00'00'", 'source': 'MCP illustrated guidebook.pdf', 'total_pages': 74, 'page': 0, 'page_label': '1'}, page_content='MCP\n2025 EDITION\nTHE ILLUSTRATED\nGUIDEBOOK \nAvi Chawla & Akshay Pachaar\nDailyDoseofDS.com\nDaily Dose of\nData Science\nFREE'),
 Document(metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250626122824Z00'00'", 'title': 'MCP book', 'moddate': "D:20250626122824Z00'00'", 'source': 'MCP illustrated guidebook.pdf', 'total_pages': 74, 'page': 1, 'page_label': '2'}, page_content='DailyDoseofDS.com \nHow to make the most out of \nthis book and your time? The reading time of this book is about 3 hours. But not all chapters will be of \nrelevance to you. This 2-minute assessment will test your current expertise and 

In [80]:
print(f"Length of docs: {len(docs)}")    # Length 74 is correct
print()

for doc in docs:
    print(doc)

Length of docs: 74

page_content='MCP
2025 EDITION
THE ILLUSTRATED
GUIDEBOOK 
Avi Chawla & Akshay Pachaar
DailyDoseofDS.com
Daily Dose of
Data Science
FREE' metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250626122824Z00'00'", 'title': 'MCP book', 'moddate': "D:20250626122824Z00'00'", 'source': 'MCP illustrated guidebook.pdf', 'total_pages': 74, 'page': 0, 'page_label': '1'}
page_content='DailyDoseofDS.com 
How to make the most out of 
this book and your time? The reading time of this book is about 3 hours. But not all chapters will be of 
relevance to you. This 2-minute assessment will test your current expertise and 
recommend chapters that will be most useful to you. 
 
Scan the QR code below or open this link to start the assessment. It will only take 
2 minutes to complete. 
 
 
https://bit.ly/mcp-assessment 
 
1' metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'cre

In [81]:
# Defining splitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

# Splitting documents into chunks
chunks = splitter.split_documents(docs)


print(f"Length of chunks: {len(chunks)}")
chunks

Length of chunks: 79


[Document(metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250626122824Z00'00'", 'title': 'MCP book', 'moddate': "D:20250626122824Z00'00'", 'source': 'MCP illustrated guidebook.pdf', 'total_pages': 74, 'page': 0, 'page_label': '1'}, page_content='MCP\n2025 EDITION\nTHE ILLUSTRATED\nGUIDEBOOK \nAvi Chawla & Akshay Pachaar\nDailyDoseofDS.com\nDaily Dose of\nData Science\nFREE'),
 Document(metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250626122824Z00'00'", 'title': 'MCP book', 'moddate': "D:20250626122824Z00'00'", 'source': 'MCP illustrated guidebook.pdf', 'total_pages': 74, 'page': 1, 'page_label': '2'}, page_content='DailyDoseofDS.com \nHow to make the most out of \nthis book and your time? The reading time of this book is about 3 hours. But not all chapters will be of \nrelevance to you. This 2-minute assessment will test your current expertise and 

In [82]:
chunks[0]

Document(metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250626122824Z00'00'", 'title': 'MCP book', 'moddate': "D:20250626122824Z00'00'", 'source': 'MCP illustrated guidebook.pdf', 'total_pages': 74, 'page': 0, 'page_label': '1'}, page_content='MCP\n2025 EDITION\nTHE ILLUSTRATED\nGUIDEBOOK \nAvi Chawla & Akshay Pachaar\nDailyDoseofDS.com\nDaily Dose of\nData Science\nFREE')

In [83]:
# Creating embedding model
embedding_model = OpenAIEmbeddings(model = 'text-embedding-3-small')

# Generating and storing embeddings in Vectpr store FAISS
vector_store = FAISS.from_documents(chunks, embedding_model)

vector_store

In [84]:
# Creating retrieval object
retrieval = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k': 4})

In [85]:
a = retrieval.invoke("Who are the authors of this book?")

a

[Document(id='0fd0ac5c-c957-4dc0-9d95-ff5a72946d4d', metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250626122824Z00'00'", 'title': 'MCP book', 'moddate': "D:20250626122824Z00'00'", 'source': 'MCP illustrated guidebook.pdf', 'total_pages': 74, 'page': 1, 'page_label': '2'}, page_content='DailyDoseofDS.com \nHow to make the most out of \nthis book and your time? The reading time of this book is about 3 hours. But not all chapters will be of \nrelevance to you. This 2-minute assessment will test your current expertise and \nrecommend chapters that will be most useful to you. \n \nScan the QR code below or open this link to start the assessment. It will only take \n2 minutes to complete. \n \n \nhttps://bit.ly/mcp-assessment \n \n1'),
 Document(id='11c3ecc6-ee66-473c-aaf8-13fb812f6d5a', metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250626122824Z00'00'

In [86]:
len(a)

4

In [87]:
# Creating a tool for RAG

@tool
def rag_tool(query):
    
    """
    This tools retrieves relevant information from the pdf.
    Use this tool when the user asks any question related to the uploaded document.
    """

    result = retrieval.invoke(query)

    context = [doc.page_content for doc in result]    # Our answer will be stored in page_content
    metadata = [doc.metadata for doc in result]    # Sometines metadata is also useful for LLM

    return {
        'query': query,
        'context': context,
        'metadata': metadata
    }

In [88]:
tools_list = [rag_tool]
model_with_tools = model.bind_tools(tools = tools_list)

In [89]:
class ChatState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

In [90]:
# Defing chat_node

def chat_node(state: ChatState):
    messages = state['messages']

    response = model_with_tools.invoke(messages)

    return {'messages': [response]}

In [91]:
# Creating a tool node
tool_node = ToolNode(tools_list)

In [92]:
# Defining graph
graph = StateGraph(ChatState)

# Adding nodes
graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

# Creating edges
graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()

In [96]:
chatbot

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [94]:
initial_state = {'messages': HumanMessage(content = "What is talked about in this book?")}

response = chatbot.invoke(initial_state)

In [95]:
response['messages'][-1].content

'The book discusses various topics related to technical writing, transcription tools, audio analysis, financial analysis, and setting up a deep researcher crew. Additionally, it provides information on how to make the most out of the book and your time.'